In [1]:
#pandas for spreadsheet processing
import pandas as pd
pd.set_option('display.max_columns', None) #display all columns

#numpy for mathematical functions
import numpy as np

#geopandas to process geospatial/gis data
import geopandas as gpd

#library for connecting to the geodatabase
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry

# User input

- All human input values will need to be input in by the user into the following cells. 
- The rest of the code will run without required human intervention.

## 1. Input Link to downloaded datasets

In [2]:
# industry by sex
# To download original dataset go to - 
# hhttps://www.nomisweb.co.uk/query/construct/components/stdListComponent.asp?menuopt=12&subcomp=100
industry_csv = r"N:\Geodatabase\Raw_Data\Census 2021\Industry\21695101657163749.csv"

## 2. Input Link to LSOA 2021 shapefile
#### To download original dataset go to - 
https://geoportal.statistics.gov.uk/datasets/ons::lower-layer-super-output-areas-december-2021-boundaries-ew-bsc-v4-2/about

In [3]:
lsoa2021_shapefile = r"C:\Users\abhimanya.achara\OneDrive - priorpartners.com\Documents\UK DATA\2021 Census Data\Lower_Layer_Super_Output_Areas_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW\Lower_Layer_Super_Output_Areas_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW.shp"

## 3. Threshold to define dominant type
This defines the % threshold from the highest value of group of data columns to define which ones will be defined as dominant. 

In [4]:
threshold_value = 5

## 1. Import & Process Data

To download original dataset go to - 
https://www.nomisweb.co.uk/query/construct/summary.asp?mode=construct&version=0&dataset=2221

In [5]:
# Create Dataframes
female_industry_df = pd.read_csv(industry_csv, skiprows = 8, nrows = 35672, skip_blank_lines = True)
male_industry_df = pd.read_csv(industry_csv, skiprows = 35703, nrows = 35672, skip_blank_lines = True)

In [6]:
# Dictionary for renaming columns with corrected keys
column_rename_map = {
    "A, B, D, E Agriculture, energy and water": "agriculture_energy_and_water",
    "C Manufacturing": "manufacturinge",
    "F Construction": "construction",
    "G, I Distribution, hotels and restaurants": "distribution_hotels_and_restaurants",
   
    "H, J Transport and communication": "transport_and_communication",
    "K, L, M, N Financial, real estate, professional and administrative activities": "finance_realestate_prof_admin_activity",
    "O, P, Q Public administration, education and health": "public_administration_education_health",
    
    "R, S, T, U Other": "other_industries",
}

# Rename columns using the dictionary
female_industry_df.rename(columns=column_rename_map, inplace=True)
male_industry_df.rename(columns=column_rename_map, inplace=True)

# Display the updated DataFrame
male_industry_df.head()

,2021 super output area - lower layer,agriculture_energy_and_water,manufacturinge,construction,distribution_hotels_and_restaurants,transport_and_communication,finance_realestate_prof_admin_activity,public_administration_education_health,other_industries
0,E01011954 : Hartlepool 001A,27,69,104,85,48,53,70,7
1,E01011969 : Hartlepool 001B,28,52,72,38,20,25,36,8
2,E01011970 : Hartlepool 001C,26,37,47,38,30,27,53,7
3,E01011971 : Hartlepool 001D,39,64,71,51,33,54,67,4
4,E01033465 : Hartlepool 001F,49,93,90,71,43,77,91,18


In [7]:
# Split the first column into two new columns
female_industry_df[['lsoa21cd', 'lsoa21nm']] = female_industry_df.iloc[:, 0].str.split(' : ', expand=True)
male_industry_df[['lsoa21cd', 'lsoa21nm']] = male_industry_df.iloc[:, 0].str.split(' : ', expand=True)

# Remove the column '2021 super output area - lower layer'
female_industry_df.drop(['2021 super output area - lower layer'], 1,  inplace=True)
male_industry_df.drop(['2021 super output area - lower layer'], 1, inplace=True)

# Rename columns for female_industry_df
cols_to_rename1 = female_industry_df.columns.difference(['lsoa21cd', 'lsoa21nm'])
female_industry_df.rename(columns={col: col.lower().replace(' ', '_') + '_female_count' for col in cols_to_rename1}, inplace=True)

# Rename columns for male_industry_df
cols_to_rename2 = male_industry_df.columns.difference(['lsoa21cd', 'lsoa21nm'])
male_industry_df.rename(columns={col: col.lower().replace(' ', '_') + '_male_count' for col in cols_to_rename2}, inplace=True)

# Remove the column '2021 super output area - lower layer'
female_industry_df.drop(['lsoa21nm'], 1,  inplace=True)
male_industry_df.drop(['lsoa21nm'], 1, inplace=True)

# Move 'lsoa21cd' and 'lsoa21nm' to the front of the dataframe
female_industry_df = female_industry_df[['lsoa21cd'] + [col for col in female_industry_df.columns if col not in ['lsoa21cd']]]
male_industry_df = male_industry_df[['lsoa21cd'] + [col for col in male_industry_df.columns if col not in ['lsoa21cd']]]

#Create sum totals
female_industry_df['total_female_industry_pop_count'] = female_industry_df.sum(axis=1,numeric_only=True)
male_industry_df['total_male_industry_pop_count'] = male_industry_df.sum(axis=1,numeric_only=True)

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_4900\1140041367.py:6: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  female_industry_df.drop(['2021 super output area - lower layer'], 1,  inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_4900\1140041367.py:7: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  male_industry_df.drop(['2021 super output area - lower layer'], 1, inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_4900\1140041367.py:18: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  female_industry_df.drop(['lsoa21nm'], 1,  inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_4900\1140041367.py:19: FutureWarning: In a future version of pandas all arguments of DataF

In [8]:
male_industry_df

,lsoa21cd,agriculture_energy_and_water_male_count,manufacturinge_male_count,construction_male_count,distribution_hotels_and_restaurants_male_count,transport_and_communication_male_count,finance_realestate_prof_admin_activity_male_count,public_administration_education_health_male_count,other_industries_male_count,total_male_industry_pop_count
0,E01011954,27,69,104,85,48,53,70,7,463
1,E01011969,28,52,72,38,20,25,36,8,279
2,E01011970,26,37,47,38,30,27,53,7,265
3,E01011971,39,64,71,51,33,54,67,4,383
4,E01033465,49,93,90,71,43,77,91,18,532
...,...,...,...,...,...,...,...,...,...,...
35667,W01001300,12,70,42,47,34,23,44,6,278
35668,W01001301,5,79,52,77,39,24,54,13,343
35669,W01001308,2,32,52,35,28,19,29,0,197
35670,W01001309,11,36,50,48,38,29,52,2,266


## 2. Feature Engineering

In [9]:
# Create percentage columns for female_industry_df
total_female_col = 'total_female_industry_pop_count'
for col in female_industry_df.columns[1:-1]:
    perc_col = col.replace('_count', '_perc')
    female_industry_df[perc_col] = (female_industry_df[col] / female_industry_df[total_female_col]) * 100

# Create percentage columns for male_industry_df
total_male_col = 'total_male_industry_pop_count'
for col in male_industry_df.columns[1:-1]:
    perc_col = col.replace('_count', '_perc')
    male_industry_df[perc_col] = (male_industry_df[col] / male_industry_df[total_male_col]) * 100

In [10]:
# Combine the two dataframes on lsoa21cd and lsoa21nm
combined_df = female_industry_df.merge(male_industry_df, on=['lsoa21cd'])

# Display the first few rows of the combined dataframe
combined_df.head()

,lsoa21cd,agriculture_energy_and_water_female_count,manufacturinge_female_count,construction_female_count,distribution_hotels_and_restaurants_female_count,transport_and_communication_female_count,finance_realestate_prof_admin_activity_female_count,public_administration_education_health_female_count,other_industries_female_count,total_female_industry_pop_count,agriculture_energy_and_water_female_perc,manufacturinge_female_perc,construction_female_perc,distribution_hotels_and_restaurants_female_perc,transport_and_communication_female_perc,finance_realestate_prof_admin_activity_female_perc,public_administration_education_health_female_perc,other_industries_female_perc,agriculture_energy_and_water_male_count,manufacturinge_male_count,construction_male_count,distribution_hotels_and_restaurants_male_count,transport_and_communication_male_count,finance_realestate_prof_admin_activity_male_count,public_administration_education_health_male_count,other_industries_male_count,total_male_industry_pop_count,agriculture_energy_and_water_male_perc,manufacturinge_male_perc,construction_male_perc,distribution_hotels_and_restaurants_male_perc,transport_and_communication_male_perc,finance_realestate_prof_admin_activity_male_perc,public_administration_education_health_male_perc,other_industries_male_perc
0,E01011954,7,18,9,134,12,52,248,14,494,1.417004,3.643725,1.821862,27.125506,2.429150,10.526316,50.202429,2.834008,27,69,104,85,48,53,70,7,463,5.831533,14.902808,22.462203,18.358531,10.367171,11.447084,15.118790,1.511879
1,E01011969,5,7,5,58,7,22,152,18,274,1.824818,2.554745,1.824818,21.167883,2.554745,8.029197,55.474453,6.569343,28,52,72,38,20,25,36,8,279,10.035842,18.637993,25.806452,13.620072,7.168459,8.960573,12.903226,2.867384
2,E01011970,3,8,4,47,6,28,137,20,253,1.185771,3.162055,1.581028,18.577075,2.371542,11.067194,54.150198,7.905138,26,37,47,38,30,27,53,7,265,9.811321,13.962264,17.735849,14.339623,11.320755,10.188679,20.000000,2.641509
3,E01011971,7,10,8,80,8,43,202,17,375,1.866667,2.666667,2.133333,21.333333,2.133333,11.466667,53.866667,4.533333,39,64,71,51,33,54,67,4,383,10.182768,16.710183,18.537859,13.315927,8.616188,14.099217,17.493473,1.044386
4,E01033465,11,21,7,86,19,56,277,19,496,2.217742,4.233871,1.411290,17.338710,3.830645,11.290323,55.846774,3.830645,49,93,90,71,43,77,91,18,532,9.210526,17.481203,16.917293,13.345865,8.082707,14.473684,17.105263,3.383459


In [11]:
# Create aggregated count columns by summing female and male counts
count_columns = [col for col in combined_df.columns if col.endswith('_female_count')]
for col in count_columns:
    base_name = col.replace('_female_count', '_count')
    male_col = col.replace('_female_count', '_male_count')
    if male_col in combined_df.columns:
        combined_df[base_name] = combined_df[col] + combined_df[male_col]

# Display the first few rows of the combined dataframe
combined_df.head()

,lsoa21cd,agriculture_energy_and_water_female_count,manufacturinge_female_count,construction_female_count,distribution_hotels_and_restaurants_female_count,transport_and_communication_female_count,finance_realestate_prof_admin_activity_female_count,public_administration_education_health_female_count,other_industries_female_count,total_female_industry_pop_count,agriculture_energy_and_water_female_perc,manufacturinge_female_perc,construction_female_perc,distribution_hotels_and_restaurants_female_perc,transport_and_communication_female_perc,finance_realestate_prof_admin_activity_female_perc,public_administration_education_health_female_perc,other_industries_female_perc,agriculture_energy_and_water_male_count,manufacturinge_male_count,construction_male_count,distribution_hotels_and_restaurants_male_count,transport_and_communication_male_count,finance_realestate_prof_admin_activity_male_count,public_administration_education_health_male_count,other_industries_male_count,total_male_industry_pop_count,agriculture_energy_and_water_male_perc,manufacturinge_male_perc,construction_male_perc,distribution_hotels_and_restaurants_male_perc,transport_and_communication_male_perc,finance_realestate_prof_admin_activity_male_perc,public_administration_education_health_male_perc,other_industries_male_perc,agriculture_energy_and_water_count,manufacturinge_count,construction_count,distribution_hotels_and_restaurants_count,transport_and_communication_count,finance_realestate_prof_admin_activity_count,public_administration_education_health_count,other_industries_count
0,E01011954,7,18,9,134,12,52,248,14,494,1.417004,3.643725,1.821862,27.125506,2.429150,10.526316,50.202429,2.834008,27,69,104,85,48,53,70,7,463,5.831533,14.902808,22.462203,18.358531,10.367171,11.447084,15.118790,1.511879,34,87,113,219,60,105,318,21
1,E01011969,5,7,5,58,7,22,152,18,274,1.824818,2.554745,1.824818,21.167883,2.554745,8.029197,55.474453,6.569343,28,52,72,38,20,25,36,8,279,10.035842,18.637993,25.806452,13.620072,7.168459,8.960573,12.903226,2.867384,33,59,77,96,27,47,188,26
2,E01011970,3,8,4,47,6,28,137,20,253,1.185771,3.162055,1.581028,18.577075,2.371542,11.067194,54.150198,7.905138,26,37,47,38,30,27,53,7,265,9.811321,13.962264,17.735849,14.339623,11.320755,10.188679,20.000000,2.641509,29,45,51,85,36,55,190,27
3,E01011971,7,10,8,80,8,43,202,17,375,1.866667,2.666667,2.133333,21.333333,2.133333,11.466667,53.866667,4.533333,39,64,71,51,33,54,67,4,383,10.182768,16.710183,18.537859,13.315927,8.616188,14.099217,17.493473,1.044386,46,74,79,131,41,97,269,21
4,E01033465,11,21,7,86,19,56,277,19,496,2.217742,4.233871,1.411290,17.338710,3.830645,11.290323,55.846774,3.830645,49,93,90,71,43,77,91,18,532,9.210526,17.481203,16.917293,13.345865,8.082707,14.473684,17.105263,3.383459,60,114,97,157,62,133,368,37


In [12]:
# Create percentage columns for female_pop_df
combined_df['total_industry_pop_count'] = combined_df['total_female_industry_pop_count'] + combined_df['total_male_industry_pop_count']
total_col = 'total_industry_pop_count'
for col in combined_df.columns[-9:-1]:
    #print(col)
    perc_col = col.replace('_count', '_perc')
    combined_df[perc_col] = (combined_df[col] / combined_df[total_col]) * 100

#combined_df.drop(['total_industry_pop_perc'],1,inplace = True)
combined_df.head()

,lsoa21cd,agriculture_energy_and_water_female_count,manufacturinge_female_count,construction_female_count,distribution_hotels_and_restaurants_female_count,transport_and_communication_female_count,finance_realestate_prof_admin_activity_female_count,public_administration_education_health_female_count,other_industries_female_count,total_female_industry_pop_count,agriculture_energy_and_water_female_perc,manufacturinge_female_perc,construction_female_perc,distribution_hotels_and_restaurants_female_perc,transport_and_communication_female_perc,finance_realestate_prof_admin_activity_female_perc,public_administration_education_health_female_perc,other_industries_female_perc,agriculture_energy_and_water_male_count,manufacturinge_male_count,construction_male_count,distribution_hotels_and_restaurants_male_count,transport_and_communication_male_count,finance_realestate_prof_admin_activity_male_count,public_administration_education_health_male_count,other_industries_male_count,total_male_industry_pop_count,agriculture_energy_and_water_male_perc,manufacturinge_male_perc,construction_male_perc,distribution_hotels_and_restaurants_male_perc,transport_and_communication_male_perc,finance_realestate_prof_admin_activity_male_perc,public_administration_education_health_male_perc,other_industries_male_perc,agriculture_energy_and_water_count,manufacturinge_count,construction_count,distribution_hotels_and_restaurants_count,transport_and_communication_count,finance_realestate_prof_admin_activity_count,public_administration_education_health_count,other_industries_count,total_industry_pop_count,agriculture_energy_and_water_perc,manufacturinge_perc,construction_perc,distribution_hotels_and_restaurants_perc,transport_and_communication_perc,finance_realestate_prof_admin_activity_perc,public_administration_education_health_perc,other_industries_perc
0,E01011954,7,18,9,134,12,52,248,14,494,1.417004,3.643725,1.821862,27.125506,2.429150,10.526316,50.202429,2.834008,27,69,104,85,48,53,70,7,463,5.831533,14.902808,22.462203,18.358531,10.367171,11.447084,15.118790,1.511879,34,87,113,219,60,105,318,21,957,3.552769,9.090909,11.807732,22.884013,6.269592,10.971787,33.228840,2.194357
1,E01011969,5,7,5,58,7,22,152,18,274,1.824818,2.554745,1.824818,21.167883,2.554745,8.029197,55.474453,6.569343,28,52,72,38,20,25,36,8,279,10.035842,18.637993,25.806452,13.620072,7.168459,8.960573,12.903226,2.867384,33,59,77,96,27,47,188,26,553,5.967450,10.669078,13.924051,17.359855,4.882459,8.499096,33.996383,4.701627
2,E01011970,3,8,4,47,6,28,137,20,253,1.185771,3.162055,1.581028,18.577075,2.371542,11.067194,54.150198,7.905138,26,37,47,38,30,27,53,7,265,9.811321,13.962264,17.735849,14.339623,11.320755,10.188679,20.000000,2.641509,29,45,51,85,36,55,190,27,518,5.598456,8.687259,9.845560,16.409266,6.949807,10.617761,36.679537,5.212355
3,E01011971,7,10,8,80,8,43,202,17,375,1.866667,2.666667,2.133333,21.333333,2.133333,11.466667,53.866667,4.533333,39,64,71,51,33,54,67,4,383,10.182768,16.710183,18.537859,13.315927,8.616188,14.099217,17.493473,1.044386,46,74,79,131,41,97,269,21,758,6.068602,9.762533,10.422164,17.282322,5.408971,12.796834,35.488127,2.770449
4,E01033465,11,21,7,86,19,56,277,19,496,2.217742,4.233871,1.411290,17.338710,3.830645,11.290323,55.846774,3.830645,49,93,90,71,43,77,91,18,532,9.210526,17.481203,16.917293,13.345865,8.082707,14.473684,17.105263,3.383459,60,114,97,157,62,133,368,37,1028,5.836576,11.089494,9.435798,15.272374,6.031128,12.937743,35.797665,3.599222


## 4. Load GIS shapefile 

In [13]:
# Attempt to read from the server first, fallback to local path if it fails
lsoa2021boundary_df = gpd.read_file(lsoa2021_shapefile)
print("Shapefile loaded successfully from the server.")

# Display the first few rows
lsoa2021boundary_df.head()

Shapefile loaded successfully from the server.


,FID,LSOA21CD,LSOA21NM,Shape__Are,Shape__Len,GlobalID,geometry
0,1,E01000001,City of London 001A,129865.314476,2635.767993,4dd060c0-a44a-4c62-aca3-b0c88c0bf0d0,"POLYGON ((532151.537 181867.433, 532152.500 18..."
1,2,E01000002,City of London 001B,228419.782242,2707.816821,d0a47b5d-5649-4242-a8e7-462078975c1d,"POLYGON ((532634.497 181926.016, 532632.048 18..."
2,3,E01000003,City of London 001C,59054.204697,1224.573160,2904cc10-e994-4a6e-b11c-06be7fbd74d2,"POLYGON ((532153.703 182165.155, 532158.250 18..."
3,4,E01000005,City of London 001E,189577.709503,2275.805344,bfcf5958-f052-4388-8efd-75bb808d8b83,"POLYGON ((533619.062 181402.364, 533639.868 18..."
4,5,E01000006,Barking and Dagenham 016A,146536.995750,1966.092607,64a24862-79c1-4c8c-ab7b-d9cec1d3c0c6,"POLYGON ((545126.852 184310.838, 545145.213 18..."


## 5. GIS data management

### Remove Rename fields

In [14]:
#Drop and rename fields
lsoa2021boundary_df.drop(['Shape__Are', 'Shape__Len', 'GlobalID'], 1, inplace = True)
lsoa2021boundary_df.rename(columns = {'LSOA21CD':'lsoa21cd','LSOA21NM':'lsoa21nm'}, inplace = True)
lsoa2021boundary_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_4900\2026090638.py:2: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa2021boundary_df.drop(['Shape__Are', 'Shape__Len', 'GlobalID'], 1, inplace = True)


,FID,lsoa21cd,lsoa21nm,geometry
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18..."
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18..."
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18..."
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18..."
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18..."


### Combine with boundary lookup table

#### LSOA21 to MSOA21 to LAD22

In [15]:
# Define the file path for msoa11 to msoa21 lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\OA21_LSOA21_MSOA21_LAD22_EW_LU.csv"

# Read the Excel file
lsoalookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop oa column
lsoalookup_df.drop(['oa21cd','lsoa21nm'],1,inplace = True)

#remove duplicates
lsoalookup_df = lsoalookup_df.drop_duplicates(subset='lsoa21cd', keep='first')

# Display the first few rows
lsoalookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_4900\1268765203.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoalookup_df.drop(['oa21cd','lsoa21nm'],1,inplace = True)


,lsoa21cd,msoa21cd,msoa21nm,lad22cd,lad22nm
0,E01000001,E02000001,City of London 001,E09000001,City of London
4,E01000003,E02000001,City of London 001,E09000001,City of London
6,E01000002,E02000001,City of London 001,E09000001,City of London
12,E01032739,E02000001,City of London 001,E09000001,City of London
13,E01032740,E02000001,City of London 001,E09000001,City of London


In [16]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, lsoalookup_df, how = 'left', on = 'lsoa21cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham


#### LSOA21 to WARD22

In [17]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Lower_Layer_Super_Output_Area_(2021)_to_Ward_(2022)_to_LAD_(2022)_Lookup_in_England_and_Wales_v3.csv"

# Read the Excel file
lsoawardlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
lsoawardlookup_df.drop(['LSOA21NM','LSOA21NMW','WD22NMW','LAD22NMW','ObjectId','LAD22CD','LAD22NM'],1,inplace = True)

# Rename fields
lsoawardlookup_df.rename(
    columns={
        'LSOA21CD': 'lsoa21cd',
        'WD22CD': 'wd22cd',
        'WD22NM': 'wd22nm',
        }, 
    inplace=True
)

# Display the first few rows
lsoawardlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_4900\1707502962.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoawardlookup_df.drop(['LSOA21NM','LSOA21NMW','WD22NMW','LAD22NMW','ObjectId','LAD22CD','LAD22NM'],1,inplace = True)


,lsoa21cd,wd22cd,wd22nm
0,E01004766,E05000650,Astley Bridge
1,E01005347,E05000722,Chadderton South
2,E01004890,E05000661,Horwich North East
3,E01004770,E05000650,Astley Bridge
4,E01004888,E05000661,Horwich North East


In [18]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, lsoawardlookup_df, how = 'inner', on = 'lsoa21cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury


#### LAD22 to REGION22

In [19]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Local_Authority_District_to_Region_(December_2022)_Lookup_in_England.csv"

# Read the Excel file
ladregionlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)

# Rename fields
ladregionlookup_df.rename(
    columns={
        'LAD22CD': 'lad22cd',
        'RGN22CD': 'rgn22cd',
        'RGN22NM': 'rgn22nm'
       }, 
    inplace=True
)

# Display the first few rows
ladregionlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_4900\4188529333.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)


,lad22cd,rgn22cd,rgn22nm
0,E06000001,E12000001,North East
1,E06000002,E12000001,North East
2,E06000003,E12000001,North East
3,E06000004,E12000001,North East
4,E06000005,E12000001,North East


In [20]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, ladregionlookup_df, how = 'left', on = 'lad22cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London


### Add data management fields

In [21]:
# Add new data management fields with specified values
lsoa2021boundary_df['data_source'] = "Census 2021 (ONS Nomis - Official Census and Labour Market Statistics)"
lsoa2021boundary_df['data_resolution'] = "LSOA 2021"
lsoa2021boundary_df['data_time_period'] = pd.to_datetime("2021-02-21")  # Convert to date format
lsoa2021boundary_df['data_web_link'] = "https://www.nomisweb.co.uk/"
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/


### Update Area

In [22]:
#Update Area in Hectares
# Ensure the dataframe has a valid geometry column
if 'geometry' in lsoa2021boundary_df.columns:
    # Calculate the area in square meters and convert to hectares (1 hectare = 10,000 m²)
    lsoa2021boundary_df['area_ha'] = lsoa2021boundary_df.geometry.area / 10_000

    # Print confirmation message
    print("Field 'area_ha' has been added and updated successfully.")
else:
    print("Error: No 'geometry' column found in lsoa2021boundary_df. Ensure it's a GeoDataFrame with valid geometries.")
    
lsoa2021boundary_df.head()

Field 'area_ha' has been added and updated successfully.


,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,14.653700


# 7. Combine Geometry and data

In [23]:
census2021_industry_lsoa2021_gdb_df = pd.merge(lsoa2021boundary_df,combined_df, how = 'left', on='lsoa21cd')
census2021_industry_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,agriculture_energy_and_water_female_count,manufacturinge_female_count,construction_female_count,distribution_hotels_and_restaurants_female_count,transport_and_communication_female_count,finance_realestate_prof_admin_activity_female_count,public_administration_education_health_female_count,other_industries_female_count,total_female_industry_pop_count,agriculture_energy_and_water_female_perc,manufacturinge_female_perc,construction_female_perc,distribution_hotels_and_restaurants_female_perc,transport_and_communication_female_perc,finance_realestate_prof_admin_activity_female_perc,public_administration_education_health_female_perc,other_industries_female_perc,agriculture_energy_and_water_male_count,manufacturinge_male_count,construction_male_count,distribution_hotels_and_restaurants_male_count,transport_and_communication_male_count,finance_realestate_prof_admin_activity_male_count,public_administration_education_health_male_count,other_industries_male_count,total_male_industry_pop_count,agriculture_energy_and_water_male_perc,manufacturinge_male_perc,construction_male_perc,distribution_hotels_and_restaurants_male_perc,transport_and_communication_male_perc,finance_realestate_prof_admin_activity_male_perc,public_administration_education_health_male_perc,other_industries_male_perc,agriculture_energy_and_water_count,manufacturinge_count,construction_count,distribution_hotels_and_restaurants_count,transport_and_communication_count,finance_realestate_prof_admin_activity_count,public_administration_education_health_count,other_industries_count,total_industry_pop_count,agriculture_energy_and_water_perc,manufacturinge_perc,construction_perc,distribution_hotels_and_restaurants_perc,transport_and_communication_perc,finance_realestate_prof_admin_activity_perc,public_administration_education_health_perc,other_industries_perc
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,0,15,5,25,50,148,94,42,379,0.000000,3.957784,1.319261,6.596306,13.192612,39.050132,24.802111,11.081794,2,10,5,26,74,251,75,41,484,0.413223,2.066116,1.033058,5.371901,15.289256,51.859504,15.495868,8.471074,2,25,10,51,124,399,169,83,863,0.231750,2.896871,1.158749,5.909618,14.368482,46.234067,19.582851,9.617613
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,0,11,1,24,37,180,71,29,353,0.000000,3.116147,0.283286,6.798867,10.481586,50.991501,20.113314,8.215297,1,9,9,25,72,294,71,42,523,0.191205,1.720841,1.720841,4.780115,13.766730,56.214149,13.575526,8.030593,1,20,10,49,109,474,142,71,876,0.114155,2.283105,1.141553,5.593607,12.442922,54.109589,16.210046,8.105023
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,1,2,6,39,43,177,110,45,423,0.236407,0.472813,1.418440,9.219858,10.165485,41.843972,26.004728,10.638298,0,9,19,61,92,257,99,40,577,0.000000,1.559792,3.292894,10.571924,15.944541,44.540728,17.157712,6.932409,1,11,25,100,135,434,209,85,1000,0.100000,1.100000,2.500000,10.000000,13.500000,43.400000,20.900000,8.500000
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Of

# 8. Final tweaks

In [24]:
# Reorder columns in the combined dataframe

final_column_order = ['FID',
 'lsoa21cd',
 'lsoa21nm',
 'geometry',
 'msoa21cd',
 'msoa21nm',
 'wd22cd',
 'wd22nm',
 'lad22cd',
 'lad22nm',
 'rgn22cd',
 'rgn22nm',
 'data_source',
 'data_resolution',
 'data_time_period',
 'data_web_link',
 'area_ha',
 'agriculture_energy_and_water_count',
 'manufacturinge_count',
 'construction_count',
 'distribution_hotels_and_restaurants_count',
 'transport_and_communication_count',
 'finance_realestate_prof_admin_activity_count',
 'public_administration_education_health_count',
 'other_industries_count',
 'total_industry_pop_count',
 'agriculture_energy_and_water_perc',
 'manufacturinge_perc',
 'construction_perc',
 'distribution_hotels_and_restaurants_perc',
 'transport_and_communication_perc',
 'finance_realestate_prof_admin_activity_perc',
 'public_administration_education_health_perc',
 'other_industries_perc',
 'agriculture_energy_and_water_female_count',
 'manufacturinge_female_count',
 'construction_female_count',
 'distribution_hotels_and_restaurants_female_count',
 'transport_and_communication_female_count',
 'finance_realestate_prof_admin_activity_female_count',
 'public_administration_education_health_female_count',
 'other_industries_female_count',
 'total_female_industry_pop_count',
 'agriculture_energy_and_water_female_perc',
 'manufacturinge_female_perc',
 'construction_female_perc',
 'distribution_hotels_and_restaurants_female_perc',
 'transport_and_communication_female_perc',
 'finance_realestate_prof_admin_activity_female_perc',
 'public_administration_education_health_female_perc',
 'other_industries_female_perc',
 'agriculture_energy_and_water_male_count',
 'manufacturinge_male_count',
 'construction_male_count',
 'distribution_hotels_and_restaurants_male_count',
 'transport_and_communication_male_count',
 'finance_realestate_prof_admin_activity_male_count',
 'public_administration_education_health_male_count',
 'other_industries_male_count',
 'total_male_industry_pop_count',
 'agriculture_energy_and_water_male_perc',
 'manufacturinge_male_perc',
 'construction_male_perc',
 'distribution_hotels_and_restaurants_male_perc',
 'transport_and_communication_male_perc',
 'finance_realestate_prof_admin_activity_male_perc',
 'public_administration_education_health_male_perc',
 'other_industries_male_perc',
]

census2021_industry_lsoa2021_gdb_df = census2021_industry_lsoa2021_gdb_df[final_column_order]

census2021_industry_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,wd22cd,wd22nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,agriculture_energy_and_water_count,manufacturinge_count,construction_count,distribution_hotels_and_restaurants_count,transport_and_communication_count,finance_realestate_prof_admin_activity_count,public_administration_education_health_count,other_industries_count,total_industry_pop_count,agriculture_energy_and_water_perc,manufacturinge_perc,construction_perc,distribution_hotels_and_restaurants_perc,transport_and_communication_perc,finance_realestate_prof_admin_activity_perc,public_administration_education_health_perc,other_industries_perc,agriculture_energy_and_water_female_count,manufacturinge_female_count,construction_female_count,distribution_hotels_and_restaurants_female_count,transport_and_communication_female_count,finance_realestate_prof_admin_activity_female_count,public_administration_education_health_female_count,other_industries_female_count,total_female_industry_pop_count,agriculture_energy_and_water_female_perc,manufacturinge_female_perc,construction_female_perc,distribution_hotels_and_restaurants_female_perc,transport_and_communication_female_perc,finance_realestate_prof_admin_activity_female_perc,public_administration_education_health_female_perc,other_industries_female_perc,agriculture_energy_and_water_male_count,manufacturinge_male_count,construction_male_count,distribution_hotels_and_restaurants_male_count,transport_and_communication_male_count,finance_realestate_prof_admin_activity_male_count,public_administration_education_health_male_count,other_industries_male_count,total_male_industry_pop_count,agriculture_energy_and_water_male_perc,manufacturinge_male_perc,construction_male_perc,distribution_hotels_and_restaurants_male_perc,transport_and_communication_male_perc,finance_realestate_prof_admin_activity_male_perc,public_administration_education_health_male_perc,other_industries_male_perc
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E05009288,Aldersgate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,2,25,10,51,124,399,169,83,863,0.231750,2.896871,1.158749,5.909618,14.368482,46.234067,19.582851,9.617613,0,15,5,25,50,148,94,42,379,0.000000,3.957784,1.319261,6.596306,13.192612,39.050132,24.802111,11.081794,2,10,5,26,74,251,75,41,484,0.413223,2.066116,1.033058,5.371901,15.289256,51.859504,15.495868,8.471074
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E05009302,Cripplegate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,1,20,10,49,109,474,142,71,876,0.114155,2.283105,1.141553,5.593607,12.442922,54.109589,16.210046,8.105023,0,11,1,24,37,180,71,29,353,0.000000,3.116147,0.283286,6.798867,10.481586,50.991501,20.113314,8.215297,1,9,9,25,72,294,71,42,523,0.191205,1.720841,1.720841,4.780115,13.766730,56.214149,13.575526,8.030593
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E05009302,Cripplegate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,1,11,25,100,135,434,209,85,1000,0.100000,1.100000,2.500000,10.000000,13.500000,43.400000,20.900000,8.500000,1,2,6,39,43,177,110,45,423,0.236407,0.472813,1.418440,9.219858,10.165485,41.843972,26.004728,10.638298,0,9,19,61,92,257,99,40,577,0.000000,1.559792,3.292894,10.571924,15.944541,44.540728,17.157712,6.932409
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E05009308,Portsoken,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Of

# 8. Create dominant field

In [26]:
def determine_dominant_group(row):
    columns = [
 'agriculture_energy_and_water_perc',
 'manufacturinge_perc',
 'construction_perc',
 'distribution_hotels_and_restaurants_perc',
 'transport_and_communication_perc',
 'finance_realestate_prof_admin_activity_perc',
 'public_administration_education_health_perc',
 'other_industries_perc',
    ]
    
    max_value = max(row[col] for col in columns)
    threshold = max_value - threshold_value
    
    dominant_groups = [col.replace('_perc', '') for col in columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_industry_lsoa2021_gdb_df['dominant_industry_group'] = census2021_industry_lsoa2021_gdb_df.apply(determine_dominant_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_industry_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,wd22cd,wd22nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,agriculture_energy_and_water_count,manufacturinge_count,construction_count,distribution_hotels_and_restaurants_count,transport_and_communication_count,finance_realestate_prof_admin_activity_count,public_administration_education_health_count,other_industries_count,total_industry_pop_count,agriculture_energy_and_water_perc,manufacturinge_perc,construction_perc,distribution_hotels_and_restaurants_perc,transport_and_communication_perc,finance_realestate_prof_admin_activity_perc,public_administration_education_health_perc,other_industries_perc,agriculture_energy_and_water_female_count,manufacturinge_female_count,construction_female_count,distribution_hotels_and_restaurants_female_count,transport_and_communication_female_count,finance_realestate_prof_admin_activity_female_count,public_administration_education_health_female_count,other_industries_female_count,total_female_industry_pop_count,agriculture_energy_and_water_female_perc,manufacturinge_female_perc,construction_female_perc,distribution_hotels_and_restaurants_female_perc,transport_and_communication_female_perc,finance_realestate_prof_admin_activity_female_perc,public_administration_education_health_female_perc,other_industries_female_perc,agriculture_energy_and_water_male_count,manufacturinge_male_count,construction_male_count,distribution_hotels_and_restaurants_male_count,transport_and_communication_male_count,finance_realestate_prof_admin_activity_male_count,public_administration_education_health_male_count,other_industries_male_count,total_male_industry_pop_count,agriculture_energy_and_water_male_perc,manufacturinge_male_perc,construction_male_perc,distribution_hotels_and_restaurants_male_perc,transport_and_communication_male_perc,finance_realestate_prof_admin_activity_male_perc,public_administration_education_health_male_perc,other_industries_male_perc,dominant_industry_group
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E05009288,Aldersgate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,2,25,10,51,124,399,169,83,863,0.231750,2.896871,1.158749,5.909618,14.368482,46.234067,19.582851,9.617613,0,15,5,25,50,148,94,42,379,0.000000,3.957784,1.319261,6.596306,13.192612,39.050132,24.802111,11.081794,2,10,5,26,74,251,75,41,484,0.413223,2.066116,1.033058,5.371901,15.289256,51.859504,15.495868,8.471074,finance_realestate_prof_admin_activity
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E05009302,Cripplegate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,1,20,10,49,109,474,142,71,876,0.114155,2.283105,1.141553,5.593607,12.442922,54.109589,16.210046,8.105023,0,11,1,24,37,180,71,29,353,0.000000,3.116147,0.283286,6.798867,10.481586,50.991501,20.113314,8.215297,1,9,9,25,72,294,71,42,523,0.191205,1.720841,1.720841,4.780115,13.766730,56.214149,13.575526,8.030593,finance_realestate_prof_admin_activity
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E05009302,Cripplegate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,1,11,25,100,135,434,209,85,1000,0.100000,1.100000,2.500000,10.000000,13.500000,43.400000,20.900000,8.500000,1,2,6,39,43,177,110,45,423,0.236407,0.472813,1.418440,9.219858,10.165485,41.843972,26.004728,10.638298,0,9,19,61,92,257,99,40,577,0.000000,1.559792,3.292894,10.571924,15.944541,44.540728,17.157712,6.932409,finance_realestate_prof_admin_activity
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.

In [27]:
def determine_dominant_group(row):
    columns = [
 'agriculture_energy_and_water_female_perc',
 'manufacturinge_female_perc',
 'construction_female_perc',
 'distribution_hotels_and_restaurants_female_perc',
 'transport_and_communication_female_perc',
 'finance_realestate_prof_admin_activity_female_perc',
 'public_administration_education_health_female_perc',
 'other_industries_female_perc',
    ]
    
    max_value = max(row[col] for col in columns)
    threshold = max_value - threshold_value
    
    dominant_groups = [col.replace('_perc', '') for col in columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_industry_lsoa2021_gdb_df['dominant_industry_group_female'] = census2021_industry_lsoa2021_gdb_df.apply(determine_dominant_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_industry_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,wd22cd,wd22nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,agriculture_energy_and_water_count,manufacturinge_count,construction_count,distribution_hotels_and_restaurants_count,transport_and_communication_count,finance_realestate_prof_admin_activity_count,public_administration_education_health_count,other_industries_count,total_industry_pop_count,agriculture_energy_and_water_perc,manufacturinge_perc,construction_perc,distribution_hotels_and_restaurants_perc,transport_and_communication_perc,finance_realestate_prof_admin_activity_perc,public_administration_education_health_perc,other_industries_perc,agriculture_energy_and_water_female_count,manufacturinge_female_count,construction_female_count,distribution_hotels_and_restaurants_female_count,transport_and_communication_female_count,finance_realestate_prof_admin_activity_female_count,public_administration_education_health_female_count,other_industries_female_count,total_female_industry_pop_count,agriculture_energy_and_water_female_perc,manufacturinge_female_perc,construction_female_perc,distribution_hotels_and_restaurants_female_perc,transport_and_communication_female_perc,finance_realestate_prof_admin_activity_female_perc,public_administration_education_health_female_perc,other_industries_female_perc,agriculture_energy_and_water_male_count,manufacturinge_male_count,construction_male_count,distribution_hotels_and_restaurants_male_count,transport_and_communication_male_count,finance_realestate_prof_admin_activity_male_count,public_administration_education_health_male_count,other_industries_male_count,total_male_industry_pop_count,agriculture_energy_and_water_male_perc,manufacturinge_male_perc,construction_male_perc,distribution_hotels_and_restaurants_male_perc,transport_and_communication_male_perc,finance_realestate_prof_admin_activity_male_perc,public_administration_education_health_male_perc,other_industries_male_perc,dominant_industry_group,dominant_industry_group_female
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E05009288,Aldersgate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,2,25,10,51,124,399,169,83,863,0.231750,2.896871,1.158749,5.909618,14.368482,46.234067,19.582851,9.617613,0,15,5,25,50,148,94,42,379,0.000000,3.957784,1.319261,6.596306,13.192612,39.050132,24.802111,11.081794,2,10,5,26,74,251,75,41,484,0.413223,2.066116,1.033058,5.371901,15.289256,51.859504,15.495868,8.471074,finance_realestate_prof_admin_activity,finance_realestate_prof_admin_activity_female
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E05009302,Cripplegate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,1,20,10,49,109,474,142,71,876,0.114155,2.283105,1.141553,5.593607,12.442922,54.109589,16.210046,8.105023,0,11,1,24,37,180,71,29,353,0.000000,3.116147,0.283286,6.798867,10.481586,50.991501,20.113314,8.215297,1,9,9,25,72,294,71,42,523,0.191205,1.720841,1.720841,4.780115,13.766730,56.214149,13.575526,8.030593,finance_realestate_prof_admin_activity,finance_realestate_prof_admin_activity_female
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E05009302,Cripplegate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,1,11,25,100,135,434,209,85,1000,0.100000,1.100000,2.500000,10.000000,13.500000,43.400000,20.900000,8.500000,1,2,6,39,43,177,110,45,423,0.236407,0.472813,1.418440,9.219858,10.165485,41.843972,26.004728,10.638298,0,9,19,61,92,257,99,40,577,0.000000,1.559792,3.292894,10.571924,15.944541,44.54072

In [28]:
def determine_dominant_group(row):
    columns = [
 'agriculture_energy_and_water_male_perc',
 'manufacturinge_male_perc',
 'construction_male_perc',
 'distribution_hotels_and_restaurants_male_perc',
 'transport_and_communication_male_perc',
 'finance_realestate_prof_admin_activity_male_perc',
 'public_administration_education_health_male_perc',
 'other_industries_male_perc',
    ]
    
    max_value = max(row[col] for col in columns)
    threshold = max_value - threshold_value
    
    dominant_groups = [col.replace('_perc', '') for col in columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_industry_lsoa2021_gdb_df['dominant_industry_group_male'] = census2021_industry_lsoa2021_gdb_df.apply(determine_dominant_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_industry_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,wd22cd,wd22nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,agriculture_energy_and_water_count,manufacturinge_count,construction_count,distribution_hotels_and_restaurants_count,transport_and_communication_count,finance_realestate_prof_admin_activity_count,public_administration_education_health_count,other_industries_count,total_industry_pop_count,agriculture_energy_and_water_perc,manufacturinge_perc,construction_perc,distribution_hotels_and_restaurants_perc,transport_and_communication_perc,finance_realestate_prof_admin_activity_perc,public_administration_education_health_perc,other_industries_perc,agriculture_energy_and_water_female_count,manufacturinge_female_count,construction_female_count,distribution_hotels_and_restaurants_female_count,transport_and_communication_female_count,finance_realestate_prof_admin_activity_female_count,public_administration_education_health_female_count,other_industries_female_count,total_female_industry_pop_count,agriculture_energy_and_water_female_perc,manufacturinge_female_perc,construction_female_perc,distribution_hotels_and_restaurants_female_perc,transport_and_communication_female_perc,finance_realestate_prof_admin_activity_female_perc,public_administration_education_health_female_perc,other_industries_female_perc,agriculture_energy_and_water_male_count,manufacturinge_male_count,construction_male_count,distribution_hotels_and_restaurants_male_count,transport_and_communication_male_count,finance_realestate_prof_admin_activity_male_count,public_administration_education_health_male_count,other_industries_male_count,total_male_industry_pop_count,agriculture_energy_and_water_male_perc,manufacturinge_male_perc,construction_male_perc,distribution_hotels_and_restaurants_male_perc,transport_and_communication_male_perc,finance_realestate_prof_admin_activity_male_perc,public_administration_education_health_male_perc,other_industries_male_perc,dominant_industry_group,dominant_industry_group_female,dominant_industry_group_male
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E05009288,Aldersgate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,2,25,10,51,124,399,169,83,863,0.231750,2.896871,1.158749,5.909618,14.368482,46.234067,19.582851,9.617613,0,15,5,25,50,148,94,42,379,0.000000,3.957784,1.319261,6.596306,13.192612,39.050132,24.802111,11.081794,2,10,5,26,74,251,75,41,484,0.413223,2.066116,1.033058,5.371901,15.289256,51.859504,15.495868,8.471074,finance_realestate_prof_admin_activity,finance_realestate_prof_admin_activity_female,finance_realestate_prof_admin_activity_male
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E05009302,Cripplegate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,1,20,10,49,109,474,142,71,876,0.114155,2.283105,1.141553,5.593607,12.442922,54.109589,16.210046,8.105023,0,11,1,24,37,180,71,29,353,0.000000,3.116147,0.283286,6.798867,10.481586,50.991501,20.113314,8.215297,1,9,9,25,72,294,71,42,523,0.191205,1.720841,1.720841,4.780115,13.766730,56.214149,13.575526,8.030593,finance_realestate_prof_admin_activity,finance_realestate_prof_admin_activity_female,finance_realestate_prof_admin_activity_male
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E05009302,Cripplegate,E09000001,City of London,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,1,11,25,100,135,434,209,85,1000,0.100000,1.100000,2.500000,10.000000,13.500000,43.400000,20.900000,8.500000,1,2,6,39,43,177,110,45,423,0.236407,0.472813,1.418440,9.219858,10.16

# 9. Upload to geodatabase

In [29]:
duplicate_columns = census2021_industry_lsoa2021_gdb_df.columns[census2021_industry_lsoa2021_gdb_df.columns.duplicated()]
print(duplicate_columns)

Index([], dtype='object')


In [30]:
# Define database connection parameters
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "uk_new"
table_name = "census2021_lsoa2021_industry"  # Desired table name
primary_key_column = "lsoa21cd"  # Define the primary key column (change based on your dataset)
geometry_column = "geometry"  # Default geometry column
# Create the database connection string (Windows Authentication - Trusted Connection)
conn_str = f"postgresql+psycopg2://@{db_host}:{db_port}/{db_name}?sslmode=disable"

# Create a SQLAlchemy engine
engine = create_engine(conn_str)

In [31]:
# Ensure the GeoDataFrame has a valid CRS before writing
if census2021_industry_lsoa2021_gdb_df.crs is None:
    print("Warning: GeoDataFrame has no CRS. Setting default to EPSG:27700 (British National Grid).")
    census2021_industry_lsoa2021_gdb_df.set_crs(epsg=27700, inplace=True)

In [33]:
# Publish the GeoDataFrame to PostGIS
census2021_industry_lsoa2021_gdb_df.to_postgis(
    name=table_name,
    con=engine,
    schema=db_schema,
    if_exists="replace",
    index=False,
    dtype={'geometry': 'POLYGON'}  # Ensure geometry is stored as MULTIPOLYGON
)

print(f"Data successfully uploaded to PostGIS: {db_schema}.{table_name}")

# Connect to the database to modify table structure
with engine.connect() as conn:
    # Set Primary Key (if it doesn't exist already)
    alter_pk_query = text(f"""
        ALTER TABLE {db_schema}.{table_name}
        ADD CONSTRAINT {table_name}_pkey PRIMARY KEY ({primary_key_column});
    """)
    
    # Create Spatial Index
    create_spatial_index_query = text(f"""
        CREATE INDEX {table_name}_geom_idx
        ON {db_schema}.{table_name}
        USING GIST ({geometry_column});
    """)

    try:
        conn.execute(alter_pk_query)  # Add Primary Key
        print(f"Primary key set on column: {primary_key_column}")
    except Exception as e:
        print(f"Could not set primary key. It may already exist. Error: {e}")

    try:
        conn.execute(create_spatial_index_query)  # Add Spatial Index
        print(f"Spatial index created for geometry column: {geometry_column}")
    except Exception as e:
        print(f"Could not create spatial index. It may already exist. Error: {e}")

print(f"GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: {db_schema}.{table_name}")

Data successfully uploaded to PostGIS: uk_new.census2021_lsoa2021_industry
Primary key set on column: lsoa21cd
Spatial index created for geometry column: geometry
GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: uk_new.census2021_lsoa2021_industry
